# FLAG 2027 - Notebook 13: VoxCeleb2 + ResNet293, then ensemble

Notebook 12: ResNet293 voice features gave -0.86 (challenge + MAV-Celeb: 27.84 -> 26.98), as much as all of VoxCeleb2 gave before. Here we combine both:

1. Re-download the two VoxCeleb2 zips, rebuild **exactly the same clips** as Notebook 07 (same selection, same Bengali filter), extract ResNet293 for them. A built-in check recomputes ECAPA and compares it with the saved features to PROVE the rows line up.
2. Train **bal75 + ResNet293** (CV, 3 seeds) -> submission ZIP + saved dev scores.
3. **Ensemble ZIP**: average of this system and Notebook 12's N4 system (raw scores of the same type; no test statistics).

First run about 2.5-3 h (mostly download + ResNet293). Resumable: finished parts are skipped. Paste the **SUMMARY** to Claude.

In [ ]:
# Cell 1 - Drive, GPU, packages, HF token
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install speechbrain huggingface_hub onnx onnx2torch
import torch, shutil
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
HF_TOKEN = userdata.get('HF_TOKEN')
print('Device:', dev, '| HF token found:', HF_TOKEN is not None, '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - paths + encoders (ECAPA for the alignment check, language-ID, ResNet293)
import os, re, json, time, zipfile, copy, subprocess
import numpy as np, pandas as pd
import librosa, soundfile as sf
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor
from huggingface_hub import hf_hub_download, list_repo_files

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features')
RESDIR = os.path.join(PROJECT, 'results')
CACHE = os.path.join(PROJECT, 'models_cache')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)

try:
    from speechbrain.inference.interfaces import Pretrained
    from speechbrain.inference.classifiers import EncoderClassifier
except ImportError:
    from speechbrain.pretrained.interfaces import Pretrained
    from speechbrain.pretrained import EncoderClassifier
class Encoder(Pretrained):
    MODULES_NEEDED = ['compute_features', 'mean_var_norm', 'embedding_model']
    def encode_batch(self, wavs, wav_lens=None):
        if wav_lens is None:
            wav_lens = torch.ones(wavs.shape[0], device=self.device)
        wavs, wav_lens = wavs.to(self.device).float(), wav_lens.to(self.device)
        feats = self.mods.mean_var_norm(self.mods.compute_features(wavs), wav_lens)
        return self.mods.embedding_model(feats, wav_lens)
ECAPA = Encoder.from_hparams(source='yangwang825/ecapa-tdnn-vox2', savedir=os.path.join(CACHE, 'ecapa_vox2'), run_opts={'device': dev}); ECAPA.eval()
LID = EncoderClassifier.from_hparams(source='speechbrain/lang-id-voxlingua107-ecapa', savedir=os.path.join(CACHE, 'lid_voxlingua107'), run_opts={'device': dev})

def batches_by_length(paths, bs):
    order = np.argsort([sf.info(p).duration for p in paths])
    for i in range(0, len(order), bs):
        yield order[i:i + bs]
def pad(paths, idx, max_sec=None):
    wavs = [librosa.load(paths[j], sr=16000, mono=True)[0] for j in idx]
    if max_sec:
        wavs = [w[:16000 * max_sec] for w in wavs]
    lens = torch.tensor([len(w) for w in wavs], dtype=torch.float); L = int(lens.max())
    b = torch.zeros(len(wavs), L)
    for k, w in enumerate(wavs):
        b[k, :len(w)] = torch.from_numpy(w)
    return b, lens / L
def enc_ecapa(paths, bs=16):
    out = np.zeros((len(paths), 192), np.float32)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc='ecapa (check)'):
        b, rl = pad(paths, idx)
        with torch.no_grad():
            out[idx] = ECAPA.encode_batch(b, rl).squeeze(1).cpu().numpy()
    return out
def language_of(paths, bs=32):
    out = [''] * len(paths)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc='language-ID'):
        b, rl = pad(paths, idx, max_sec=10)
        with torch.no_grad():
            labs = LID.classify_batch(b.to(dev), rl.to(dev))[3]
        for j, lab in zip(idx, labs):
            out[j] = lab.split(':')[0].strip()
    return np.array(out)
def enc_r293(paths, sec=6, bs=32):
    # identical to Notebook 12 (ONNX -> PyTorch, fixed 6-s crops)
    import torchaudio
    from onnx2torch import convert
    repo = 'Wespeaker/wespeaker-voxceleb-resnet293-LM'
    onnx_name = [f for f in list_repo_files(repo) if f.endswith('.onnx')][0]
    net = convert(hf_hub_download(repo, onnx_name, local_dir=os.path.join(CACHE, 'wespeaker_r293'))).to(dev).eval()
    L = 16000 * sec
    def fbank(p):
        w = librosa.load(p, sr=16000, mono=True)[0]
        if len(w) >= L:
            s0 = (len(w) - L) // 2; w = w[s0:s0 + L]
        else:
            w = np.tile(w, int(np.ceil(L / max(len(w), 1))))[:L]
        fb = torchaudio.compliance.kaldi.fbank(torch.from_numpy(w).unsqueeze(0) * (1 << 15), num_mel_bins=80, frame_length=25,
                                               frame_shift=10, dither=0.0, sample_frequency=16000, window_type='hamming', use_energy=False)
        return fb - fb.mean(0, keepdim=True)
    out = []
    for i in tqdm(range(0, len(paths), bs), desc='resnet293'):
        x = torch.stack([fbank(p) for p in paths[i:i + bs]]).to(dev)
        with torch.no_grad():
            e = net(x)
        e = e[0] if isinstance(e, (list, tuple)) else e
        out.append(e.reshape(len(x), -1).float().cpu().numpy())
    return np.concatenate(out)
print('encoders ready')

In [ ]:
# Cell 3 - rebuild the SAME VoxCeleb2 clips as Notebook 07 and extract ResNet293 (RESUMABLE per zip)
ZIPS = [('vox2/vox2_test_mp4.zip', 40), ('vox2/vox2_mp4_1.zip', 20)]   # (zip, CLIPS_PER_ID used in Notebook 07)
def ffmpeg_audio(job):
    mp4, wav, jpg = job
    a = subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-i', mp4, '-vn', '-ac', '1', '-ar', '16000', wav]).returncode
    v = subprocess.run(['ffmpeg', '-loglevel', 'error', '-y', '-ss', '1.0', '-i', mp4, '-frames:v', '1', '-q:v', '2', jpg]).returncode
    return a == 0 and v == 0 and os.path.exists(wav) and os.path.exists(jpg)   # same success rule as Notebook 07

for zname, cpi in ZIPS:
    tag = os.path.basename(zname).replace('.zip', '')
    old_path = os.path.join(FEATDIR, 'vox_' + tag + '.npz')
    out_path = os.path.join(FEATDIR, 'vox_' + tag + '_r293.npy')
    if os.path.exists(out_path):
        print(tag, 'ResNet293 already extracted'); continue
    OLD = dict(np.load(old_path))
    t0 = time.time()
    zpath = hf_hub_download('ProgramComputer/voxceleb', zname, repo_type='dataset', token=HF_TOKEN, local_dir='/content/vox')
    zf = zipfile.ZipFile(zpath)
    tree = {}
    for n in zf.namelist():
        mm = re.search(r'(id\d{5})/([^/]+)/([^/]+\.mp4)$', n)
        if mm:
            tree.setdefault(mm.group(1), {}).setdefault(mm.group(2), []).append(n)
    rng = np.random.default_rng(0)
    picked = []
    for pid, vids in tree.items():
        pools = [list(rng.permutation(v)) for v in vids.values()]
        chosen = []
        while len(chosen) < cpi and any(pools):
            for p in pools:
                if p and len(chosen) < cpi:
                    chosen.append(p.pop())
        picked += [(pid, c) for c in chosen]
    W = '/content/vox_work/' + tag; os.makedirs(W, exist_ok=True)
    jobs = []
    for i, (pid, member) in enumerate(picked):
        jobs.append((zf.extract(member, W + '/mp4'), '%s/%06d.wav' % (W, i), '%s/%06d.jpg' % (W, i)))
    zf.close(); os.remove(zpath)
    with ThreadPoolExecutor(8) as ex:
        ok = list(tqdm(ex.map(ffmpeg_audio, jobs), total=len(jobs), desc='ffmpeg'))
    keep = [i for i, g in enumerate(ok) if g]
    wavs = [jobs[i][1] for i in keep]; spk = np.array(['vox_' + picked[i][0] for i in keep])
    lang = language_of(wavs); nb = lang != 'bn'
    wavs = [w for w, k in zip(wavs, nb) if k]; spk = spk[nb]
    # ALIGNMENT CHECK: same speakers in the same order + recomputed ECAPA == saved ECAPA
    assert len(spk) == len(OLD['spk']) and (spk == OLD['spk']).all(), 'row order differs from Notebook 07 (%d vs %d rows)' % (len(spk), len(OLD['spk']))
    probe = np.random.default_rng(1).choice(len(wavs), 300, replace=False)
    e_new = enc_ecapa([wavs[j] for j in probe]); e_old = OLD['ecapa'][probe]
    cos = (e_new * e_old).sum(1) / (np.linalg.norm(e_new, axis=1) * np.linalg.norm(e_old, axis=1) + 1e-9)
    note('%s: %d rows | alignment check: median cosine %.4f (should be ~1.0)' % (tag, len(spk), np.median(cos)))
    assert np.median(cos) > 0.98, 'rows do not line up - stop and tell Claude'
    r = enc_r293(wavs).astype(np.float32)
    np.save(out_path, r)
    shutil.rmtree(W, ignore_errors=True)
    note('%s: ResNet293 extracted for %d clips in %.0f min' % (tag, len(r), (time.time() - t0) / 60))

In [ ]:
# Cell 4 - load everything: challenge + MAV + VOX with CLIP | ECAPA + WavLM + ResNet293
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
LOCAL = '/content/flag_small'; os.makedirs(LOCAL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    flag = os.path.join(LOCAL, z + '.done')
    if not os.path.exists(flag):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(LOCAL, members=[m for m in zf.namelist() if m.endswith(('.csv', '.txt'))])
        open(flag, 'w').close()
INDEX = [os.path.join(r, f) for r, _, fs in os.walk(LOCAL) for f in fs if f.endswith(('.csv', '.txt'))]
def find_one(*parts):
    hits = [p for p in INDEX if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FEAT = {}
Xv, y = read_feat(find_one('train_English_voices.csv'), 192)
FEAT['ecapa'] = {'train': Xv}
for s in SPLIT_NAMES[1:]:
    track, lang = s.rsplit('_', 1)
    FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv', 'r293']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files}
y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
VOX = {k: [] for k in ['spk', 'clip', 'ecapa', 'wavlm_sv', 'r293']}
for zname, _ in ZIPS:
    tag = os.path.basename(zname).replace('.zip', '')
    d = dict(np.load(os.path.join(FEATDIR, 'vox_' + tag + '.npz')))
    d['r293'] = np.load(os.path.join(FEATDIR, 'vox_' + tag + '_r293.npy'))
    for k in VOX:
        VOX[k].append(d[k])
VOX = {k: np.concatenate(v) for k, v in VOX.items()}

def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
VOICE = ['ecapa', 'wavlm_sv', 'r293']
def face_of(clip):
    return l2(clip).astype(np.float32)
def voice_of(blocks):
    return np.concatenate([l2(b) for b in blocks], 1).astype(np.float32)
N_CH, N_MAV, N_VOX = len(y), len(MAV['spk']), len(VOX['spk'])
ext_spk = list(MAV['spk']) + list(VOX['spk'])
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.concatenate([y, np.array([ext_lab[s] for s in ext_spk])])
D_ALL = np.r_[np.zeros(N_CH, int), np.ones(N_MAV, int), 2 * np.ones(N_VOX, int)]
F_RAW = np.concatenate([face_of(FEAT['clip']['train']), face_of(MAV['clip']), face_of(VOX['clip'])])
V_RAW = np.concatenate([voice_of([FEAT[n]['train'] for n in VOICE]), voice_of([FEAT['r293']['mav'] if n == 'r293' else MAV[n] for n in VOICE]),
                        voice_of([VOX[n] for n in VOICE])])
EXT = N_CH + np.arange(N_MAV + N_VOX)
note('Table: %d rows, %d people, face dim %d, voice dim %d' % (len(Y_ALL), len(set(Y_ALL)), F_RAW.shape[1], V_RAW.shape[1]))

In [ ]:
# Cell 5 - bal75 recipe (Notebook 08) + CV
def domain_normalize(tr_idx):
    F, V = F_RAW.copy(), V_RAW.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_RAW[trd].mean(0), F_RAW[trd].std(0) + 1e-6, V_RAW[trd].mean(0), V_RAW[trd].std(0) + 1e-6)
        F[alld] = (F_RAW[alld] - st[0]) / st[1]; V[alld] = (V_RAW[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused)
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm
    ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed)
    spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))

SEEDS = [1, 2, 3]
P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS, IN_FRAC = 16, 8, 2.0, 1e-4, 100, 15, 60, 0.75
def train_variant(tr_idx, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    all_tr = np.concatenate([tr_idx, EXT])
    F, V, ch = domain_normalize(all_tr)
    XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev)
    cls_in = np.zeros(len(cl), bool); cls_in[np.unique(ytr_np[D_ALL[all_tr] < 2])] = True
    in_cls, out_cls = np.where(cls_in)[0], np.where(~cls_in)[0]; n_in = int(round(P_ID * IN_FRAC))
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev)
    model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6)
    mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01)
    best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = np.r_[rng.choice(in_cls, n_in, replace=False), rng.choice(out_cls, P_ID - n_in, replace=False)]
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j])
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch

NAME = 'B1_bal75_r293'
CSV = os.path.join(RESDIR, 'nb13_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['variant', 'seed', 'fold'])
t0 = time.time()
for seed in SEEDS:
    for k in range(K):
        if ((done.variant == NAME) & (done.seed == seed) & (done.fold == k)).any():
            continue
        tr_idx, vsets = FOLD_DATA[k]
        model, best_ep, XF, XV, _ = train_variant(tr_idx, vsets, seed=seed)
        e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
        row = pd.DataFrame([dict(variant=NAME, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
        row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)
        done = pd.concat([done, row])
        print('%s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min' % (NAME, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))
R = pd.read_csv(CSV); R = R[R.variant == NAME]
ps = R.groupby('seed')[['std', 'gender']].mean()
note('B1 bal75 + ResNet293 (CV 5-fold x 3 seeds): std %.2f | gender %.2f | overall %.2f   (bal75 without r293 = 26.89; N4 = 26.98)'
     % (ps['std'].mean(), ps['gender'].mean(), (ps['std'].mean() + ps['gender'].mean()) / 2))

In [ ]:
# Cell 6 - final B1 model -> ZIP + dev scores; then ENSEMBLE ZIP (B1 + Notebook 12's N4, average of raw scores)
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
def write_zip(name, scores):   # scores: split -> HIGHER = same; files get the distance-like direction (LOWER = same)
    OUT = '/content/sub_' + name
    for track in ['no_gender', 'gender']:
        os.makedirs(os.path.join(OUT, track), exist_ok=True)
        for lang, fname in LANG_FILE.items():
            split = track + '_' + lang
            pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang + '_test.txt')) if l.strip()]
            assert len(pair_ids) == len(scores[split]) and np.isfinite(scores[split]).all()
            with open(os.path.join(OUT, track, fname), 'w') as f:
                for pid, v in zip(pair_ids, -scores[split]):
                    f.write('%s %.6f\n' % (pid, v))
    zp = os.path.join(SUBDIR, name + '.zip')
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for track in ['no_gender', 'gender']:
            for fname in LANG_FILE.values():
                zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
    note('Submission ZIP saved: ' + zp[len(BASE):])

ep = int(np.median(R['best_ep']))
finals = []
for s in SEEDS:
    m, _, _, _, ch = train_variant(np.arange(N_CH), None, seed=s, max_epochs=ep)
    finals.append(m)
B1 = {}
for track in ['no_gender', 'gender']:
    for lang in LANG_FILE:
        split = track + '_' + lang
        fx = ((face_of(FEAT['clip'][split]) - ch[0]) / ch[1]).astype(np.float32)          # challenge TRAIN statistics
        vx = ((voice_of([FEAT[n][split] for n in VOICE]) - ch[2]) / ch[3]).astype(np.float32)
        xf = torch.tensor(fx, device=dev); xv = torch.tensor(vx, device=dev)
        B1[split] = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
np.savez(os.path.join(RESDIR, 'devscores_13_B1.npz'), **B1)
write_zip('13_B1_bal75_r293', B1)

N4 = dict(np.load(os.path.join(RESDIR, 'devscores_12_N4.npz')))
ENS = {s: (B1[s] + N4[s]) / 2 for s in B1}      # same score type (-L2 of tanh embeddings) -> plain average, no test statistics
write_zip('13_ENS_B1+N4', ENS)
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)